# 3.1 — Inspect embedding vectors

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM


model_id = "HuggingFaceTB/SmolLM2-135M-Instruct"

# CPU/GPU cmpatibility
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("")

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id).to(device)
model.eval()

text = "I am polishing my AI Engineeering"

inputs = tokenizer(
    text,
    return_tensors="pt",
    add_special_tokens=False,
).to(device)

embedding_layer = model.get_input_embeddings()

with torch.inference_mode():
    embeddings = embedding_layer(inputs["input_ids"])

print("Device:", device)
print("Embedding matrix:", embedding_layer.weight.shape)
print("Input IDs:", inputs["input_ids"].shape)
print("Embedding output:", embeddings.shape)
print("First token vector:", embeddings[0, 0, :5])

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Device: cpu
Embedding matrix: torch.Size([49152, 576])
Input IDs: torch.Size([1, 7])
Embedding output: torch.Size([1, 7, 576])
First token vector: tensor([-0.0496,  0.0386,  0.0972, -0.1318,  0.0859], dtype=torch.bfloat16)


# 3.2 — Contextual Embeddings


### Same token, different meanings: static vs contextual embeddings

```text
                    Same token ID ("bank")  →  Same initial embedding
                                          │
                ┌─────────────────────────┴─────────────────────────┐
                ▼                                                   ▼
   Money context                                       River context
   "I deposited cash at the bank"                      "We sat on the river bank"
                │                                                   │
                ▼                                                   ▼
          Transformer                                         Transformer
                │                                                   │
                ▼                                                   ▼
      Contextual vector A                                 Contextual vector B
      (finance meaning)                                   (riverside meaning)
```

The embedding table is just a lookup, so `bank` always starts from the same vector, whatever sentence it's in.
Self-attention then mixes in information from the surrounding words, so by the last layer the two `bank` vectors point in different directions: one near *money*, the other near *river*.

In [5]:
sentences = [
    "I deposited money at the bank",
    "I walked beside the river bank"
]

initial_vectors = []
contextual_vectors = []
last_token_ids = []

for sentence in sentences:
    inputs = tokenizer(sentence, return_tensors="pt").to(device)

    with torch.inference_mode():
        initial = model.get_input_embeddings()(
            inputs["input_ids"]
        )

        outputs = model(
            **inputs,
            output_hidden_states=True
        )

    last_token_ids.append(inputs["input_ids"][0, -1].item())
    initial_vectors.append(initial[0, -1].cpu())
    contextual_vectors.append(outputs.hidden_states[-1][0, -1].cpu())

print("Last token IDs:", last_token_ids)

print(
    "Initial embeddings identical:",
    torch.allclose(initial_vectors[0], initial_vectors[1])
)

print(
    "Contextual representations identical:",
    torch.allclose(contextual_vectors[0], contextual_vectors[1])
)



Last token IDs: [5461, 5461]
Initial embeddings identical: True
Contextual representations identical: False


# 3.3 Mean pooling

In [6]:
sentence = "I am learning AI"

inputs = tokenizer(
    sentence,
    return_tensors="pt"
).to(device)

with torch.inference_mode():
    outputs = model(
        **inputs,
        output_hidden_states=True
    )

token_vectors = outputs.hidden_states[-1]

sentence_vector = token_vectors.mean(dim=1)

print("Token vectors:", token_vectors.shape)
print("Sentence vector:", sentence_vector.shape)

Token vectors: torch.Size([1, 4, 576])
Sentence vector: torch.Size([1, 576])


# 3.3 — Mean Pooling vs. Last-Token Pooling

- **Mean pooling:** Averages all token representations.



- **Last-token pooling:** Selects the final token's contextual representation. In a causal Transformer like SmolLM2, this representation has access to the preceding tokens.

In [7]:
mean_vector = token_vectors.mean(dim=1)

last_token_vector = token_vectors[:, -1, :]

print("Mean pooling:", mean_vector.shape)
print("Last-token pooling:", last_token_vector.shape)

print("Vectors idential:", torch.allclose(mean_vector, last_token_vector))

Mean pooling: torch.Size([1, 576])
Last-token pooling: torch.Size([1, 576])
Vectors idential: False


# 3.4: Cosine Similarity vs. Dot Product

- **Dot product** measures the combined alignment and magnitude of two vectors.

- **Cosine similarity** measures the direction of two vectors relative to each other.

In [8]:
import torch.nn.functional as F

a = torch.tensor([3.0, 4.0], device=device)
b = torch.tensor([4.0, 3.0], device=device)

a_norm = F.normalize(a, dim=0)
b_norm = F.normalize(b, dim=0)

print("Dot product:", torch.dot(a_norm, b_norm).item())
print("Cosine similarity:", F.cosine_similarity(a, b, dim=0).item())

Dot product: 0.9600000381469727
Cosine similarity: 0.9600000381469727


In [11]:
import torch
import torch.nn.functional as F

query = torch.tensor([1.0, 0.0], device=device)

# print(query.shape)
# print(query.unsqueeze(0).shape)
docs = torch.tensor([
    [0.8, 0.6],
    [3.0, 4.0]
], device=device)

docs = torch.tensor([
    [0.8, 0.6],
    [3.0, 4.0]
], device=device)

# Raw dot product
dot_score = docs @ query

# Cosine similarity
cosine_score = F.cosine_similarity(
    docs, query.unsqueeze(0), dim=1
)

# Dot product after L2 normalization
normalized_dot = (
    F.normalize(docs, dim=1)
    @ F.normalize(query, dim=0)
)

print("Dot:", dot_score)
print("Cosine:", cosine_score)
print("Normalized dot:", normalized_dot)

Dot: tensor([0.8000, 3.0000])
Cosine: tensor([0.8000, 0.6000])
Normalized dot: tensor([0.8000, 0.6000])


# 3.5: Retrieval and Recall@

- **Recall@k** measures how many relevant documents appear in the top k retrieved results.

- For example, if two documents are relevant and only one appears in the top two results, Recall@2 = 0.5.

In [14]:
query = "How does tokenization work in language models?"

passages = [
    "Tokenization splits text into smaller subword units.",
    "BPE repeatedly merges frequent adjacent token pairs.",
    "The weather is sunny today.",
    "GPUs accelerate matrix operations."
]

relevant_ids = {0, 1}

def embed(text):
    inputs = tokenizer(text, return_tensors="pt").to(device)

    with torch.inference_mode():
        outputs = model(**inputs, output_hidden_states=True)

    # Mean poolinga
    return outputs.hidden_states[-1].mean(dim=1).squeeze(0)

query_vector = embed(query)
print(query_vector.shape)
doc_vectors = torch.stack([embed(p) for p in passages])

# Raw dot-product similarity
dot_scores = doc_vectors @ query_vector

# L2-normalized dot product = cosine similarity
normalized_docs = F.normalize(doc_vectors, dim=1)
normalized_query = F.normalize(query_vector, dim=0)

cosine_scores = normalized_docs @ normalized_query

# Rank documents by cosine similarity
top_k = 2
ranked_ids = torch.argsort(cosine_scores, descending=True)
retrieved_ids = ranked_ids[:top_k].tolist()

recall_at_2 = len(set(retrieved_ids) & relevant_ids) / len(relevant_ids)

print("Query shape:", query_vector.shape)
print("Documents shape:", doc_vectors.shape)

for i, passage in enumerate(passages):
    print(
        f"Doc {i}: dot={dot_scores[i].item():.4f}, "
        f"cosine={cosine_scores[i].item():.4f} | {passage}"
    )

print("Top 2 document IDs:", retrieved_ids)
print("Recall@2:", recall_at_2)

torch.Size([576])
Query shape: torch.Size([576])
Documents shape: torch.Size([4, 576])
Doc 0: dot=1112.0000, cosine=0.8281 | Tokenization splits text into smaller subword units.
Doc 1: dot=1112.0000, cosine=0.8086 | BPE repeatedly merges frequent adjacent token pairs.
Doc 2: dot=792.0000, cosine=0.6641 | The weather is sunny today.
Doc 3: dot=1248.0000, cosine=0.8633 | GPUs accelerate matrix operations.
Top 2 document IDs: [3, 0]
Recall@2: 0.5


# 3.6 — Matrix Dimensions and Broadcasting

In [17]:
# Existing embeddings from Section 3.5
print("Documents:", doc_vectors.shape)
print("Query:", query_vector.shape)

# Matrix-vector multiplication
scores = doc_vectors @ query_vector
print("Scores:", scores.shape)

# Broadcasting example
expanded_query = query_vector.unsqueeze(0)

print("Expanded query:", expanded_query.shape)
print("Broadcasted addition:",
      (doc_vectors + expanded_query).shape)

Documents: torch.Size([4, 576])
Query: torch.Size([576])
Scores: torch.Size([4])
Expanded query: torch.Size([1, 576])
Broadcasted addition: torch.Size([4, 576])


In [21]:
doc_vectors.shape


torch.Size([4, 576])